In [ ]:
import os
# Set NEAFE_DATA_FILE to the original station workbook before execution.


In [ ]:
import pandas as pd
import numpy as np
import time
from tsfresh.feature_extraction import extract_features, MinimalFCParameters  # ← 改成 Minimal，最轻量
from tsfresh.utilities.dataframe_functions import roll_time_series
from tsfresh import select_features
from tsfresh.feature_extraction.settings import from_columns
from tqdm import tqdm
import warnings
import multiprocessing
warnings.filterwarnings('ignore')

def main():
    print("开始 TSFresh 特征工程（最大加速组合版：Minimal + 小窗口 + step + top-k 特征）...")
    print(f"可用 CPU 核心数: {multiprocessing.cpu_count()}（将自动使用全部）\n")

    # ==============================
    # 1. 读取数据
    # ==============================
    data_path = os.environ["NEAFE_DATA_FILE"]
    data = pd.read_excel(data_path, sheet_name="Sheet2")
    data = data.dropna().reset_index(drop=True)

    datetime_col = data.columns[0]
    target_col = data.columns[-1]

    data[datetime_col] = pd.to_datetime(data[datetime_col])
    print("原始数据形状:", data.shape)

    # ==============================
    # 2. 自动检测采样频率并设置窗口大小 → 改成 2 小时（方式1）
    # ==============================
    data = data.sort_values(datetime_col).reset_index(drop=True)
    time_deltas = data[datetime_col].diff().dropna()
    median_delta_sec = time_deltas.dt.total_seconds().median()
    samples_per_hour = 3600 / median_delta_sec
    WINDOW_SIZE = int(2 * samples_per_hour)  # ← 方式1：改成 2 小时
    print(f"检测到采样间隔: {median_delta_sec:.0f}秒 ≈ {samples_per_hour:.1f}样本/小时")
    print(f"窗口大小设置为: {WINDOW_SIZE} 条记录（≈2小时）")

    # ==============================
    # 3. 数据划分 (60/20/20)
    # ==============================
    split_idx_1 = int(len(data) * 0.6)
    split_idx_2 = int(len(data) * 0.8)

    train_df = data.iloc[:split_idx_1].copy()
    valid_df = data.iloc[split_idx_1:split_idx_2].copy()
    test_df = data.iloc[split_idx_2:].copy()

    print(f"Train: {train_df.shape} | Valid: {valid_df.shape} | Test: {test_df.shape}")

    # ==============================
    # 4. 只在训练集上 roll + extract（方式2 + 方式4）
    # ==============================
    start_fe = time.time()

    train_data = train_df.copy()
    train_data["id"] = 1

    print("正在训练集上 rolling + 提取 Minimal 特征模板（仅用于特征选择）...")
    train_rolled = roll_time_series(
        train_data,
        column_id="id",
        column_sort=datetime_col,
        max_timeshift=WINDOW_SIZE - 1,
        min_timeshift=WINDOW_SIZE - 1,
        rolling_direction=1  # ← 方式2：每 5 步滑动一次，大幅减少窗口数量
    )

    fc_parameters = MinimalFCParameters()  # ← 方式4：最轻量的参数集，特征很少，速度最快

    train_features_template = extract_features(
        train_rolled,
        column_id="id",
        column_sort=datetime_col,
        default_fc_parameters=fc_parameters,
        n_jobs=0,
        disable_progressbar=False
    )

    print(f"训练集模板共生成 {train_features_template.shape[1]} 个特征")

    # 特征选择
    print("正在进行特征选择...")
    
    y_train_aligned = train_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True)
    
    # 关键修复：让 y 的 index 与 X 完全相同
    y_train_aligned.index = train_features_template.index
    
    # （可选）打印确认
    print("X index 类型和前几个示例:", type(train_features_template.index[0]), train_features_template.index[:3].tolist())
    print("y index 已对齐为:", y_train_aligned.index[:3].tolist())
    
    selected = select_features(train_features_template, y_train_aligned, ml_task='regression')
    good_cols = selected.columns.tolist()
    print(f"特征选择后保留 {len(good_cols)} 个有效特征")
    
    # 如果想截取 top-k
    good_cols = good_cols[:30]
    kind_to_fc_parameters_selected = from_columns(good_cols)
    print(f"截取 top 30 后使用 {len(good_cols)} 个特征")

    # ==============================
    # 5. 逐点计算函数（保持不变）
    # ==============================
    def compute_tsfresh_history(df, window_size, kind_to_fc_parameters, good_cols, desc="逐点计算"):
        df = df.reset_index(drop=True)
        results = []
        all_expected_cols = good_cols
        start_loop = time.time()
        processed = 0

        for i in tqdm(range(window_size - 1, len(df)), desc=desc):
            window = df.iloc[i - window_size + 1 : i + 1].copy()
            window["id"] = 1

            try:
                feat = extract_features(
                    window,
                    column_id="id",
                    column_sort=datetime_col,
                    kind_to_fc_parameters=kind_to_fc_parameters,
                    default_fc_parameters=None,
                    n_jobs=0,
                    disable_progressbar=True,
                    show_warnings=False
                )
                feat = feat.reindex(columns=all_expected_cols).fillna(np.nan)
                feat.index = [df.index[i]]
                results.append(feat)
            except Exception as e:
                print(f"警告：窗口 {i} 提取失败 → {e}")
                empty_row = pd.DataFrame(np.nan, index=[df.index[i]], columns=all_expected_cols)
                results.append(empty_row)

            processed += 1

        if not results:
            raise ValueError("没有成功提取任何特征！")

        return pd.concat(results)

    # ==============================
    # 6. 逐点计算三个集合
    # ==============================
    print("\n正在为训练集逐点计算...")
    train_ft = compute_tsfresh_history(train_df, WINDOW_SIZE, kind_to_fc_parameters_selected, good_cols, desc="Train")

    print("\n正在为验证集逐点计算...")
    valid_ft = compute_tsfresh_history(valid_df, WINDOW_SIZE, kind_to_fc_parameters_selected, good_cols, desc="Valid")

    print("\n正在为测试集逐点计算...")
    test_ft = compute_tsfresh_history(test_df, WINDOW_SIZE, kind_to_fc_parameters_selected, good_cols, desc="Test")

    end_fe = time.time()
    print(f"\nTSFresh 特征工程完成！总用时: {end_fe - start_fe:.1f} 秒")
    print(f"最终特征维度: {train_ft.shape[1]}（已丢弃前 {WINDOW_SIZE-1} 条）")

    # ==============================
    # 7. 清洗 + 保存（y 保存方式优化）
    # ==============================
    for df in [train_ft, valid_ft, test_ft]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

    train_mean = train_ft.mean()
    train_ft = train_ft.fillna(train_mean)
    valid_ft = valid_ft.fillna(train_mean)
    test_ft = test_ft.fillna(train_mean)

    train_ft = train_ft.select_dtypes(include=[np.number])
    valid_ft = valid_ft.select_dtypes(include=[np.number])
    test_ft = test_ft.select_dtypes(include=[np.number])

    train_x = train_df.drop(columns=[target_col]).select_dtypes(include=[np.number]).iloc[WINDOW_SIZE-1:].reset_index(drop=True)
    valid_x = valid_df.drop(columns=[target_col]).select_dtypes(include=[np.number]).iloc[WINDOW_SIZE-1:].reset_index(drop=True)
    test_x = test_df.drop(columns=[target_col]).select_dtypes(include=[np.number]).iloc[WINDOW_SIZE-1:].reset_index(drop=True)

    train_x.to_feather("train_raw_wind6.feather")
    valid_x.to_feather("valid_raw_wind6.feather")
    test_x.to_feather("test_raw_wind6.feather")

    train_ft.reset_index(drop=True).to_feather("train_raw_TSFreshFCP_wind6.feather")
    valid_ft.reset_index(drop=True).to_feather("valid_raw_TSFreshFCP_wind6.feather")
    test_ft.reset_index(drop=True).to_feather("test_raw_TSFreshFCP_wind6.feather")

    # y 保存方式优化（避免列名问题）
    train_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True).to_frame(name=target_col).to_feather("train_y_wind6.feather")
    valid_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True).to_frame(name=target_col).to_feather("valid_y_wind6.feather")
    test_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True).to_frame(name=target_col).to_feather("test_y_wind6.feather")

    print("\n=== 全部保存完成！===")

if __name__ == '__main__':
    start_fe = time.time()
    main()
    end_fe = time.time()
    print(end_fe-start_fe)

In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Raw
train_raw = pd.read_feather("train_raw_wind6.feather")
valid_raw = pd.read_feather("valid_raw_wind6.feather")
test_raw  = pd.read_feather("test_raw_wind6.feather")

# tsfresh
train_neafe = pd.read_feather("train_raw_TSFreshFCP_wind6.feather")
valid_neafe = pd.read_feather("valid_raw_TSFreshFCP_wind6.feather")
test_neafe  = pd.read_feather("test_raw_TSFreshFCP_wind6.feather")

# Label
train_y = pd.read_feather("train_y_wind6.feather").values.ravel()
valid_y = pd.read_feather("valid_y_wind6.feather").values.ravel()
test_y  = pd.read_feather("test_y_wind6.feather").values.ravel()

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    nrmse = rmse / (y_true.max() - y_true.min())
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "MSE": mse,
        "nRMSE": nrmse,
        "R2": r2
    }

In [ ]:
from xgboost import XGBRegressor

def run_xgb(train_x, valid_x, test_x, tag):

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "random_state": 42,
            "n_jobs": 4,
            "tree_method": "hist"  
        }

        model = XGBRegressor(**params)

        model.fit(
            train_x, train_y,
            eval_set=[(valid_x, valid_y)],
            verbose=False
        )

        pred = model.predict(valid_x)
        return np.sqrt(mean_squared_error(valid_y, pred))


    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10)

    best_params = study.best_params
    best_params.update({
        "random_state": 42,
        "n_jobs": 4,
        "tree_method": "hist"
    })

    # 重新训练
    best_model = XGBRegressor(**best_params)

    best_model.fit(
        train_x, train_y,
        eval_set=[(valid_x, valid_y)],
        verbose=False
    )

    preds = best_model.predict(test_x)
    metrics = evaluate(test_y, preds)
    
    print("Best Params:", best_params)
    print("Test Metrics:", metrics)
    
    return metrics 

In [ ]:
xgb_raw = run_xgb(train_raw, valid_raw, test_raw, "Raw")
xgb_ts = run_xgb(train_neafe, valid_neafe, test_neafe, "ts")
print("xgb Raw:", xgb_raw)
print("xgb tsfresh:", xgb_ts)